# 04 · 自己实现 Decision Transformer（含训练）

DT 把强化学习变成序列建模：把轨迹排成
`[想要的回报 R, 状态 s, 动作 a, R, s, a, ...]`，训练一个小 Transformer 预测下一个动作。
推理时把"想要的回报"设高一点，模型就（理论上）输出能达到该回报的动作。

CPU 训练本教学版约 1-2 分钟。我们平台上训的完整版（16 维状态、2 万步）也只要 CPU 10 分钟。

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caitq2024/auto_auction/blob/main/notebooks/04_%E8%87%AA%E5%B7%B1%E5%AE%9E%E7%8E%B0DT.ipynb)

> Colab 用户先运行下面的数据下载 cell；本地运行可跳过。


In [ ]:
# Colab 环境准备：拉取教学数据（本地运行且 data/ 已存在时自动跳过）
import os, urllib.request
os.makedirs('data', exist_ok=True)
for f in ['period7_adv0.csv.gz', 'period7_tick0_adv0to7.csv.gz']:
    if not os.path.exists(f'data/{f}'):
        urllib.request.urlretrieve(f'https://github.com/caitq2024/auto_auction/raw/main/notebooks/data/{f}', f'data/{f}')
        print('downloaded', f)


In [ ]:
import pandas as pd, numpy as np, torch, torch.nn as nn
torch.manual_seed(0)
df = pd.read_csv('data/period7_adv0.csv.gz')
BUDGET, NUM_TICK = float(df.budget.iloc[0]), 48

# 轨迹: 每时段 (return-to-go, state, action)
rows, remaining = [], BUDGET
for tick, g in df.sort_values('timeStepIndex').groupby('timeStepIndex'):
    spent = g.cost[g.isExposed==1].sum()
    state  = [(NUM_TICK-tick)/NUM_TICK, remaining/BUDGET, g.pValue.mean()*1000]
    action = g.bid.mean()/max(g.pValue.mean(),1e-9) / 100
    reward = g[g.isExposed==1].pValue.sum()
    rows.append([state, action, reward]); remaining -= spent
rtg = np.cumsum([r[2] for r in rows][::-1])[::-1].copy()  # return-to-go
S = torch.tensor([r[0] for r in rows], dtype=torch.float32)
A = torch.tensor([[r[1]] for r in rows], dtype=torch.float32)
R = torch.tensor(rtg, dtype=torch.float32).unsqueeze(-1) / 30.0  # 缩放

In [ ]:
class TinyDT(nn.Module):
    """K=8 上下文窗口的最小 DT：embed(R,s,a) -> TransformerEncoder -> 预测 a"""
    def __init__(self, d=64, K=8):
        super().__init__()
        self.K = K
        self.emb_r, self.emb_s, self.emb_a = nn.Linear(1,d), nn.Linear(3,d), nn.Linear(1,d)
        self.pos = nn.Embedding(K, d)
        layer = nn.TransformerEncoderLayer(d, 4, 128, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, 2)
        self.head = nn.Linear(d, 1)
    def forward(self, r, s, a):
        T = r.shape[1]
        pos = self.pos(torch.arange(T))
        # 交织 [R,s,a] 三 token；预测每步动作时只看当前 R,s 与历史
        x = torch.stack([self.emb_r(r)+pos, self.emb_s(s)+pos, self.emb_a(a)+pos], 2).flatten(1,2)
        mask = torch.triu(torch.ones(3*T, 3*T), 1).bool()
        h = self.tr(x, mask=mask)
        return self.head(h[:, 1::3])  # 每组第2个位置（看过 R,s）预测 a

model = TinyDT()
opt = torch.optim.Adam(model.parameters(), 1e-3)
K = 8
for step in range(600):
    i = np.random.randint(0, NUM_TICK - K)
    r, s, a = R[i:i+K][None], S[i:i+K][None], A[i:i+K][None]
    a_in = torch.cat([torch.zeros(1,1,1), a[:,:-1]], 1)  # teacher forcing，右移
    pred = model(r, s, a_in)
    loss = ((pred - a)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print('final loss:', float(loss))

In [ ]:
# 推理：设一个"想要的回报"，滚动生成 alpha 序列
@torch.no_grad()
def rollout_alphas(target_return=25.0):
    rs, ss, as_ = [torch.tensor([[target_return/30]])], [S[:1]], [torch.zeros(1,1)]
    alphas = []
    for t in range(NUM_TICK):
        r = torch.stack(rs[-K:], 1).reshape(1, -1, 1)
        s = torch.cat(ss[-K:]).unsqueeze(0)
        a = torch.stack(as_[-K:], 1).reshape(1, -1, 1)
        pred = model(r, s, a)[0, -1]
        alphas.append(float(pred) * 100)
        # 教学简化：用日志状态推进（真实评估要接 replay/模拟器）
        if t+1 < NUM_TICK:
            rs.append(rs[-1]); ss.append(S[t+1:t+2]); as_.append(pred[None])
    return alphas

alphas = rollout_alphas(25.0)
import matplotlib.pyplot as plt
plt.plot(alphas); plt.title('DT 生成的 alpha 序列 (target return=25)')

## 思考题

1. 把 `target_return` 从 25 调到 50，alpha 序列变激进了吗？为什么"要求更高回报"不一定真能拿到？
2. 我们平台的实测：DT 在三种数据口径下都垫底（模仿了 PID 主导数据的坏习惯）——
   你能从"训练目标 = 模仿数据分布"推出这个结果吗？
3. 进阶：把 02 篇的 replay 接进来做真实评估（注意 state 里的预算要用 replay 的实时值）。